# VAE 논문 수식 해독 × PyTorch 실습

**학습 목표**: 주변우도와 ELBO를 수식으로 유도하고, `mu`, `logvar`, `diag`, `reparameterization`, `reconstruction`, `KL`이 어떤 텐서 연산인지 직접 확인한다.

**진행 순서**: 개념 → 확률분포 실험 → gradient 확인 → KL 수치 검증 → 실제 VAE 학습 → 이미지 복원·생성 → 퀴즈.

> 실행: `Run All`. 먼저 MNIST 다운로드를 시도하며 네트워크나 torchvision을 쓸 수 없으면 scikit-learn의 내장 손글씨 숫자 데이터(8×8)로 자동 대체한다. CPU도 가능하다. PyTorch, matplotlib, numpy가 필요하고, 백업 데이터에는 scikit-learn이 필요하다.

## 1. 논문 기호 사전

| 수식 | 의미 | 코드 |
|---|---|---|
| $x$ | 관측 이미지 | `x` |
| $z$ | 숨겨진 잠재변수 | `z` |
| $p(z)=\mathcal N(0,I)$ | 생성 prior | `torch.randn(...)` |
| $q_\phi(z\mid x)$ | 인코더가 만든 근사 사후분포 | `mu, logvar = encoder(x)` |
| $p_\theta(x\mid z)$ | 디코더의 이미지 조건부 분포 | `decoder(z)` |
| $\log p_\theta(x)$ | 로그 주변우도, latent를 적분 제거 | 직접 계산하기 어려움 |
| $\mathrm{diag}(\sigma^2)$ | 대각 공분산 행렬 | 각 latent 차원의 `logvar` |

**확률을 평가**: $p_\theta(x\mid z)$ / **샘플링**: $x\sim p_\theta(\cdot\mid z)$.

여기서는 각 픽셀에 *독립 Bernoulli 분포*를 가정한다. 따라서 디코더의 출력은 픽셀별 로짓이며, `binary_cross_entropy_with_logits`가 **negative log-likelihood**에 대응한다. 실제 0~1 픽셀 값은 엄밀히 Bernoulli 관측이 아니므로 교육용 관례에 따라 BCE를 사용하는 근사적 설정이다.

## 2. ELBO 유도

$$\begin{aligned}
\log p_\theta(x)
&=\log\int p_\theta(x,z)\,dz\\
&=\log\mathbb E_{q_\phi(z\mid x)}\left[\frac{p_\theta(x,z)}{q_\phi(z\mid x)}\right]\\
&\ge \mathbb E_{q_\phi}\left[\log\frac{p_\theta(x,z)}{q_\phi(z\mid x)}\right] &&\text{(Jensen)}\\
&=\mathbb E_{q_\phi}[\log p_\theta(x\mid z)]-D_{\rm KL}(q_\phi(z\mid x)\|p(z))\\
&=\operatorname{ELBO}(x).
\end{aligned}$$

**최소화할 loss**는 $-\mathrm{ELBO}=\underbrace{-\mathbb E_q\log p_\theta(x\mid z)}_{\text{reconstruction NLL}}+\underbrace{D_{\rm KL}(q_\phi(z\mid x)\|p(z))}_{\text{KL regularization}}$.

정확한 identity: $\log p_\theta(x)-\mathrm{ELBO}=D_{\rm KL}(q_\phi(z\mid x)\|p_\theta(z\mid x))\ge0$. 유도에서 posterior $p_\theta(z\mid x)$는 *prior* $p(z)$와 다르다.

In [ ]:
import math, random
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

## 3. 왜 `logvar`, 왜 `diag`인가?

$\sigma^2 = \exp(\mathrm{logvar})$, $\sigma=\exp(\tfrac12\mathrm{logvar})$.

$z\in\mathbb R^d$의 대각 공분산은 $\Sigma=\mathrm{diag}(\sigma_1^2,\ldots,\sigma_d^2)$이고 서로 다른 좌표 사이의 공분산이 0이라는 뜻이다. **다변량 가우시안에서는** 좌표들 사이의 독립성과 동치다.

In [ ]:
mu = torch.tensor([1.0, -1.0, 0.0])
logvar = torch.log(torch.tensor([0.25, 1.0, 4.0]))
variance = torch.exp(logvar)
std = torch.exp(0.5 * logvar)
covariance = torch.diag(variance)
print("mu:", mu)
print("logvar:", logvar)
print("variance:", variance)
print("std:", std)
print("diag covariance:\n", covariance)
assert torch.allclose(std.square(), variance)

## 4. Reparameterization: 똑같은 분포지만 미분 경로가 다르다

$z\sim\mathcal N(\mu_\phi(x),\mathrm{diag}(\sigma_\phi^2(x)))$를 $\epsilon\sim\mathcal N(0,I)$, $z=\mu_\phi(x)+\sigma_\phi(x)\odot\epsilon$으로 바꾼다.

$8\epsilon$을 고정했을 때$9 $\partial z/\partial\mu=1$, $\partial z/\partial\mathrm{logvar}=\tfrac12\sigma\odot\epsilon$. 단순 `.sample()`과 `.rsample()`을 비교해보자.

In [ ]:
mu_grad = torch.tensor([0.4], requires_grad=True)
logvar_grad = torch.tensor([-0.3], requires_grad=True)
std_grad = torch.exp(0.5 * logvar_grad)
dist = torch.distributions.Normal(mu_grad, std_grad)

# sample()은 보통 미분 경로를 추적하지 않음
z_sample = dist.sample()
print("sample requires_grad:", z_sample.requires_grad)

# rsample()은 reparameterized sample
z_rsample = dist.rsample()
loss_demo = z_rsample.square().sum()
loss_demo.backward()
print("rsample requires_grad:", z_rsample.requires_grad)
print("dLoss/dmu:", mu_grad.grad.item())
print("dLoss/dlogvar:", logvar_grad.grad.item())
assert mu_grad.grad is not None and logvar_grad.grad is not None

## 5. 대각 가우시안 KL을 코드로 읽기

$$D_{\rm KL}\left(\mathcal N(\mu,\mathrm{diag}(\sigma^2))\|\mathcal N(0,I)\right)=\frac12\sum_{j=1}^d(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1)$$

`logvar=log(σ²)`이므로 $\sigma^2=\exp(\mathrm{logvar})$ 대입하면 바로 코드가 된다.

In [ ]:
def kl_standard_normal(mu, logvar):
    # [batch, latent_dim] -> [batch]
    return 0.5 * (mu.square() + logvar.exp() - logvar - 1).sum(dim=1)

mu_test = torch.tensor([[0., 0.], [1., -0.5]])
logvar_test = torch.tensor([[0., 0.], [0.2, -0.4]])
manual = kl_standard_normal(mu_test, logvar_test)
q = torch.distributions.Independent(
    torch.distributions.Normal(mu_test, torch.exp(0.5*logvar_test)), 1)
p = torch.distributions.Independent(
    torch.distributions.Normal(torch.zeros_like(mu_test), torch.ones_like(mu_test)), 1)
library = torch.distributions.kl_divergence(q, p)
print("직접 구현:", manual)
print("라이브러리:", library)
assert torch.allclose(manual, library, atol=1e-6)
assert abs(manual[0].item()) < 1e-7

## 6. 손글씨 숫자 데이터 로딩

가능하면 **MNIST (28×28)** 를 사용한다. 접속이 불가능하면 자동으로 scikit-learn digits (8×8) 데이터로 대체한다. 어느 쪽이든 학습 코드는 동일하고 `input_dim`만 바뀐다.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

try:
    from torchvision import datasets, transforms
    data = datasets.MNIST(root="./data", train=True, download=True,
                          transform=transforms.ToTensor())
    loader = DataLoader(data, batch_size=128, shuffle=True)
    image_side = 28
    data_name = "MNIST"
except Exception as exc:
    print("MNIST 로딩 실패 -> 내장 digits 데이터 사용:", str(exc)[:180])
    from sklearn.datasets import load_digits
    digits = load_digits()
    pixels = torch.tensor(digits.images / 16.0, dtype=torch.float32).unsqueeze(1)
    data = TensorDataset(pixels, torch.tensor(digits.target))
    loader = DataLoader(data, batch_size=128, shuffle=True)
    image_side = 8
    data_name = "sklearn digits"
input_dim = image_side**2
x_demo, y_demo = next(iter(loader))
print(data_name, '| x:', tuple(x_demo.shape), '| min/max:', (x_demo.min().item(), x_demo.max().item()))
fig, axes = plt.subplots(1, 6, figsize=(10,2))
for ax, img in zip(axes, x_demo[:6]):
    ax.imshow(img[0], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle(data_name); plt.show()

## 7. VAE 모델 구현

- Encoder: $x\mapsto(\mu_\phi(x),\mathrm{logvar}_\phi(x))$.
- Reparameterize: $z=\mu+\exp(\tfrac12\mathrm{logvar})\odot\epsilon$.
- Decoder: $z\mapsto$ 각 픽셀에 대한 **Bernoulli 로짓**.
- Decoder 출력은 이미지 샘플 그 자체가 아니라 **$p_\theta(x\mid z)$를 정의하는 파라미터**다.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=2, hidden_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x.flatten(1))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        return self.decoder(z)  # Bernoulli logits, not probabilities

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar, z

model = VAE(input_dim, latent_dim=2).to(device)
logits, mu, logvar, z = model(x_demo[:8].to(device))
print('x:', x_demo[:8].shape, '| mu:', mu.shape, '| logvar:', logvar.shape)
print('z:', z.shape, '| logits:', logits.shape)
assert logits.shape == (8, input_dim) and z.shape == (8, 2)

## 8. Negative ELBO Loss 정의

$$-\mathrm{ELBO}\approx \underbrace{-\log p_\theta(x\mid z)}_{\mathrm{reconstruction\;NLL}}+\underbrace{D_{\rm KL}(q_\phi(z\mid x)\|p(z))}_{\mathrm{KL}}$$

기댓값은 매 샘플 당 $z$ 하나를 뽑은 **Monte Carlo 추정**으로 근사한다. 픽셀별 NLL은 합산하고 KL도 잠재차원에 대해 합산한 다음, 배치에 대해서만 평균을 취한다. 이렇게 하면 원래의 per-image ELBO 단위와 대응돼.

In [ ]:
def vae_loss(logits, x, mu, logvar):
    x_flat = x.flatten(1)
    recon_per_item = F.binary_cross_entropy_with_logits(
        logits, x_flat, reduction="none"
    ).sum(dim=1)
    kl_per_item = kl_standard_normal(mu, logvar)
    total_per_item = recon_per_item + kl_per_item
    return total_per_item.mean(), recon_per_item.mean(), kl_per_item.mean()

loss, recon, kl = vae_loss(logits, x_demo[:8].to(device), mu, logvar)
print(f'total={loss.item():.3f}, reconstruction={recon.item():.3f}, KL={kl.item():.3f}')
assert torch.allclose(loss, recon + kl, atol=1e-5)

## 9. Gradient가 실제로 인코더에 도달하는지 확인

`loss.backward()`는 그래디언트를 계산해 parameter의 `.grad`에 누적한다. `optimizer.step()`에서 **파라미터를 업데이트**한다. 둘은 다른 과정이다.

In [ ]:
model.zero_grad(set_to_none=True)
logits, mu, logvar, z = model(x_demo[:8].to(device))
loss, recon, kl = vae_loss(logits, x_demo[:8].to(device), mu, logvar)
loss.backward()
for name in ('fc_mu.weight', 'fc_logvar.weight', 'decoder.0.weight'):
    grad = dict(model.named_parameters())[name].grad
    print(name, '| grad_norm=', grad.norm().item())
    assert grad is not None and torch.isfinite(grad).all()

## 10. 실제 학습 (기본 3 epoch)

학습 결과는 무작위 초기값, 데이터셋과 학습 환경에 따라 달라진다. 실행 시간을 줄이려면 `epochs=1`로 바꿔도 된다.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 3
history = []
for epoch in range(epochs):
    model.train()
    sums = np.zeros(3); n_seen = 0
    for x, _ in loader:
        x = x.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits, mu, logvar, z = model(x)
        total, recon, kl = vae_loss(logits, x, mu, logvar)
        total.backward()
        optimizer.step()
        sums += np.array([total.item(), recon.item(), kl.item()]) * x.size(0)
        n_seen += x.size(0)
    means = sums/n_seen
    history.append(means)
    print(f'Epoch {epoch+1}: total={means[0]:.3f}, recon={means[1]:.3f}, KL={means[2]:.3f}')

plt.plot(np.array(history)[:,0], label='negative ELBO')
plt.plot(np.array(history)[:,1], label='reconstruction NLL')
plt.plot(np.array(history)[:,2], label='KL')
plt.xlabel('epoch'); plt.ylabel('nats / image'); plt.legend(); plt.show()

## 11. 복원과 샘플링은 어떻게 다른가?

**복원:** 관측한 $x\to q_\phi(z\mid x)\to z\to p_\theta(x\mid z)$.

**생성:** $z\sim p(z)=\mathcal N(0,I)$ 후 디코더로 생성. 아래에서는 Bernoulli 분포의 픽셀 확률(평균)을 시각화한다. 실제 이진 샘플을 얻으려면 그 확률로 픽셀별 베르누이 샘플링을 수행한다.

In [ ]:
model.eval()
with torch.no_grad():
    x, _ = next(iter(loader))
    x = x[:8].to(device)
    logits, mu, logvar, z = model(x)
    x_recon = torch.sigmoid(logits).reshape(-1, 1, image_side, image_side)
    z_prior = torch.randn(8, model.latent_dim, device=device)
    x_new = torch.sigmoid(model.decode(z_prior)).reshape(-1, 1, image_side, image_side)
fig, axes = plt.subplots(3, 8, figsize=(12,5))
for row, batch in enumerate((x, x_recon, x_new)):
    for j in range(8):
        axes[row,j].imshow(batch[j,0].cpu(), cmap='gray', vmin=0, vmax=1)
        axes[row,j].axis('off')
for row, label in enumerate(('Original', 'Reconstruction', 'Prior sample')):
    axes[row,0].set_ylabel(label)
plt.tight_layout();plt.show()

## 12. 직접 풀어볼 문제

1. `z = mu + std * eps`에서 `std.detach()`를 넣으면 reconstruction loss가 `logvar`에 전달하는 gradient 경로는 어떻게 달라질까? KL 항의 gradient는 여전히 존재할까?
2. `kl_standard_normal`에서 `mu=0`, `logvar=0`이면 왜 KL이 0이 되는지 전개해라.
3. `p(z)`를 표준정규분포로 정한 상태에서 `mu=[2, 0]`, `logvar=[0, 0]`일 때 KL 값을 계산해라.
4. `reduction='mean'`으로 픽셀 손실을 평균하면 KL과의 상대적인 규모가 어떻게 바뀌는지 설명해라.
5. 모델이 내놓는 `logits`를 `torch.sigmoid`로 바꾸는 것은 확률 **평가**인가, 실제 이미지 **샘플링**인가?

**점검할 구분**: probability vs likelihood, $p(z)$ vs $p_\theta(z\mid x)$, $q_\phi(z\mid x)$ vs 단일 샘플 $z$, gradient 계산 vs optimizer 업데이트.

### 더 깊게 읽을 수식

$$\mathbb E_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)] = \mathbb E_{\epsilon\sim\mathcal N(0,I)}[\log p_\theta(x\mid \mu_\phi(x)+\sigma_\phi(x)\odot\epsilon)]$$

이 등식이 바로 **reparameterization으로 기대값의 샘플링 분포를 파라미터와 분리하는 구조**다. 다음 훈련에서는 이 식의 $\phi$에 대한 미분을 직접 전개해볼 수 있다.